# Eclipse — HalfKAv2 NNUE Eval Training

Trains the Eclipse chess engine's NNUE on Lichess Stockfish depth-22 cp-eval labels.

**Inputs**
* Kaggle Dataset: `eclipse-partial` containing `eval_training.txt.gz`.
  Lines are `<fen>;<score_cp>` — Stockfish depth-22 evaluation, 415M positions,
  1800+ Elo, 180s+ TC.
* Hardware: GPU T4 ×2 (Settings → Accelerator → GPU T4 x2).

**Output**
* `/kaggle/working/halfkav2.pt` — PyTorch state_dict (HalfKAv2-1024x2-512-128-1).
  Download locally and convert with `scripts/convert_halfkav2_nnue.py from-torch`.

**Source of truth**
This notebook embeds `scripts/train_halfkav2.py`. When the trainer changes, update both.


In [ ]:
# Environment check
import sys, platform
import torch
print(f'python   {platform.python_version()}')
print(f'torch    {torch.__version__}')
print(f'cuda     {torch.cuda.is_available()} ({torch.version.cuda})')
if torch.cuda.is_available():
    print(f'gpu      {torch.cuda.get_device_name(0)}')
    print(f'vram     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Locate eval_training data anywhere under /kaggle/input/.
# Kaggle decompresses .gz files on dataset upload, so check both extensions.
# _open_data() in the next cell handles both formats transparently.
import os
from pathlib import Path

matches = (sorted(Path('/kaggle/input').glob('**/eval_training.txt.gz')) or
           sorted(Path('/kaggle/input').glob('**/eval_training.txt')))
if not matches:
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        print('  ' * depth + os.path.basename(root) + '/')
        for fname in files:
            print('  ' * (depth + 1) + fname)
    raise FileNotFoundError(
        'eval_training.txt(.gz) not found under /kaggle/input. '
        'Attach the eclipse-partial dataset in Notebook settings.')

DATA_PATH = matches[0]
size_mb = DATA_PATH.stat().st_size / 1e6
print(f'training data: {DATA_PATH} ({size_mb:.1f} MB)')

In [ ]:
# HalfKAv2 feature indexing -- exact mirror of src/nnue.cpp::feature_index
# and scripts/train_halfkav2.py. Drift here = silently wrong inference.
import gzip, math, random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

FT_IN_FEATURES = 45056          # HalfKAv2: 64 (king sq) * 64 (piece sq) * 11 (piece slots)
FT_OUT         = 1024           # keep in sync with kFtOutSize in src/nnue.hpp
L1_IN          = 2 * FT_OUT     # 2048 (concat of both perspectives)
L1_OUT         = 512
L2_OUT         = 128
L3_OUT         = 1              # Value head: single logit (sigmoid = win prob)
N_PIECE_SLOTS  = 11

_PIECE_LETTER_TO_TYPE = {'p':1, 'n':2, 'b':3, 'r':4, 'q':5, 'k':6}
_FLIP_RANK = lambda s: s ^ 56

def parse_fen_board(fen_board):
    pieces = []
    sq = 56
    for ch in fen_board:
        if ch == '/':
            sq -= 16
            continue
        if ch.isdigit():
            sq += int(ch)
            continue
        is_white = ch.isupper()
        pt = _PIECE_LETTER_TO_TYPE[ch.lower()]
        pieces.append((sq, 0 if is_white else 1, pt))
        sq += 1
    return pieces

def feature_index(king_sq, piece_sq, pt, piece_is_ours):
    # ours: P=0,N=1,B=2,R=3,Q=4 (own king skipped). theirs: P=5..K=10.
    if piece_is_ours and pt == 6:
        raise ValueError('own king is not a HalfKAv2 feature')
    slot = (pt - 1) if piece_is_ours else (5 + (pt - 1))
    return king_sq * (64 * N_PIECE_SLOTS) + piece_sq * N_PIECE_SLOTS + slot

def fen_to_features(fen):
    parts = fen.split()
    board, stm = parts[0], parts[1]
    pieces = parse_fen_board(board)
    stm_color = 0 if stm == 'w' else 1
    king_sq = [None, None]
    for sq, color, pt in pieces:
        if pt == 6:
            king_sq[color] = sq
    if king_sq[0] is None or king_sq[1] is None:
        raise ValueError(f'FEN missing a king: {fen}')
    indices = [[], []]
    for persp in (0, 1):
        ksq = king_sq[persp] if persp == 0 else _FLIP_RANK(king_sq[persp])
        for sq, color, pt in pieces:
            is_ours = (color == persp)
            if is_ours and pt == 6:
                continue
            psq = sq if persp == 0 else _FLIP_RANK(sq)
            indices[persp].append(feature_index(ksq, psq, pt, is_ours))
    return (indices[0], indices[1]) if stm_color == 0 else (indices[1], indices[0])

In [ ]:
# Dataset helpers — _open_data handles .gz transparently; _parse_line is shared.
# HalfKAv2Dataset: in-memory, for the val set (≤ a few million positions).
# HalfKAv2StreamDataset: reservoir-shuffle streaming, for the full training set.
# HalfKAv2BinaryDataset: memmap random-access, used after preprocessing.
import struct as _struct
import torch
from torch.utils.data import DataLoader, Dataset, IterableDataset
MAX_ACTIVE = 32

def _open_data(path):
    if str(path).endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8')
    return open(path, encoding='utf-8')

def _parse_line(line, cp_scale):
    line = line.strip()
    if not line or line.startswith('#'):
        return None
    parts = line.split(';')
    if len(parts) == 4:
        try:
            fen      = parts[0].strip()
            win_prob = float(parts[1]) + 0.5 * float(parts[2])
        except ValueError:
            return None
    elif len(parts) == 2:
        try:
            fen      = parts[0].strip()
            win_prob = 1.0 / (1.0 + math.exp(-float(parts[1]) / cp_scale))
        except ValueError:
            return None
    else:
        return None
    try:
        us, them = fen_to_features(fen)
    except (KeyError, IndexError, ValueError):
        return None
    import numpy as _np
    us_arr   = _np.full(MAX_ACTIVE, FT_IN_FEATURES, dtype=_np.int32)
    them_arr = _np.full(MAX_ACTIVE, FT_IN_FEATURES, dtype=_np.int32)
    us_arr[:min(len(us),   MAX_ACTIVE)] = us[:MAX_ACTIVE]
    them_arr[:min(len(them), MAX_ACTIVE)] = them[:MAX_ACTIVE]
    return us_arr, them_arr, _np.float32(win_prob)

class HalfKAv2Dataset(Dataset):
    """In-memory dataset. Used for the val set (first val_size lines)."""
    def __init__(self, path, max_lines=None, cp_scale=410.0):
        import numpy as _np
        n = max_lines or self._count(path)
        if n == 0:
            raise RuntimeError(f'no rows in {path}')
        self.us_idx   = _np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        self.them_idx = _np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        self.targets  = _np.zeros((n, 1), dtype=_np.float32)
        i = 0
        with _open_data(path) as f:
            for line in f:
                if i >= n: break
                p = _parse_line(line, cp_scale)
                if p is None: continue
                self.us_idx[i], self.them_idx[i], self.targets[i, 0] = p
                i += 1
        if i < n:
            self.us_idx   = self.us_idx[:i]
            self.them_idx = self.them_idx[:i]
            self.targets  = self.targets[:i]
        print(f'val: loaded {i} positions')
    @staticmethod
    def _count(path):
        n = 0
        with _open_data(path) as f:
            for l in f:
                s = l.strip()
                if s and not s.startswith('#'): n += 1
        return n
    def __len__(self):  return len(self.targets)
    def __getitem__(self, i): return self.us_idx[i], self.them_idx[i], self.targets[i]

class HalfKAv2StreamDataset(IterableDataset):
    """Streaming dataset with reservoir shuffle buffer.
    Memory per worker: shuffle_buffer * (32+32)*4 + shuffle_buffer*4 bytes (~133 MB at 500k).
    max_lines is the TOTAL cap across all workers; each worker gets max_lines // n_workers.
    """
    def __init__(self, path, skip_lines=0, max_lines=None,
                 cp_scale=410.0, shuffle_buffer=500_000):
        self.path, self.skip_lines = path, skip_lines
        self.max_lines, self.cp_scale = max_lines, cp_scale
        self.shuffle_buffer = shuffle_buffer
    def __iter__(self):
        import numpy as _np, random as _rnd, torch as _torch
        wi = _torch.utils.data.get_worker_info()
        wid, nw = (wi.id, wi.num_workers) if wi else (0, 1)
        per_worker_max = ((self.max_lines + nw - 1) // nw
                          if self.max_lines is not None else None)
        buf_us   = _np.full((self.shuffle_buffer, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        buf_them = _np.full((self.shuffle_buffer, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        buf_tgt  = _np.zeros(self.shuffle_buffer, dtype=_np.float32)
        fill = 0; emitted = 0; didx = 0
        with _open_data(self.path) as f:
            if self.skip_lines:
                sk = 0
                for raw in f:
                    if raw.strip() and not raw.strip().startswith('#'):
                        sk += 1
                        if sk >= self.skip_lines: break
            for raw in f:
                if per_worker_max is not None and emitted >= per_worker_max: break
                stripped = raw.strip()
                if not stripped or stripped.startswith('#'): continue
                # Stride check BEFORE parsing: only parse lines this worker owns.
                if didx % nw != wid: didx += 1; continue
                didx += 1
                p = _parse_line(raw, self.cp_scale)
                if p is None: continue
                us_a, th_a, wp = p
                if fill < self.shuffle_buffer:
                    buf_us[fill]=us_a; buf_them[fill]=th_a; buf_tgt[fill]=wp; fill+=1
                else:
                    idx = _rnd.randrange(fill)
                    yield buf_us[idx].copy(), buf_them[idx].copy(), buf_tgt[idx:idx+1].copy()
                    buf_us[idx]=us_a; buf_them[idx]=th_a; buf_tgt[idx]=wp; emitted+=1
        perm = list(range(fill)); _rnd.shuffle(perm)
        for idx in perm:
            if per_worker_max is not None and emitted >= per_worker_max: break
            yield buf_us[idx].copy(), buf_them[idx].copy(), buf_tgt[idx:idx+1].copy()
            emitted += 1

# Binary format constants — must match preprocess_halfkav2.py
_BIN_MAGIC  = b'HKAV2BIN'
_BIN_HSIZE  = 16
_BIN_DTYPE  = np.dtype([('us',  np.uint16, (MAX_ACTIVE,)),
                         ('them',np.uint16, (MAX_ACTIVE,)),
                         ('tgt', np.float16)])

class HalfKAv2BinaryDataset(Dataset):
    """Memmap dataset from preprocess_halfkav2.py. Zero FEN parsing.
    Workers do array indexing only — use DataLoader(shuffle=True)."""
    def __init__(self, path, start=0, end=None):
        with open(path, 'rb') as f:
            if f.read(8) != _BIN_MAGIC: raise ValueError(f'{path}: not HKAV2BIN')
            n_total = _struct.unpack('<q', f.read(8))[0]
        self._mm  = np.memmap(path, dtype=_BIN_DTYPE, mode='r',
                              offset=_BIN_HSIZE, shape=(n_total,))
        self.start = start
        self.end   = n_total if end is None else min(end, n_total)
    def __len__(self): return self.end - self.start
    def __getitem__(self, i):
        r = self._mm[self.start + i]
        return (r['us'].astype(np.int64),
                r['them'].astype(np.int64),
                np.array([float(r['tgt'])], dtype=np.float32))

def collate(batch):
    import numpy as _np
    us   = _np.stack([b[0] for b in batch])
    them = _np.stack([b[1] for b in batch])
    tgt  = _np.stack([b[2] for b in batch])
    return (torch.from_numpy(us).long(),
            torch.from_numpy(them).long(),
            torch.from_numpy(tgt))

In [ ]:
# ── Preprocessing: text → binary (run once; idempotent) ──────────────────────
# Converts eval_training.txt to a 130-byte/record binary file (uint16 feature
# indices + float16 target). Workers do array indexing instead of FEN parsing,
# eliminating the CPU bottleneck.
#
# 140M records ≈ 18 GB — fits in /kaggle/working (20 GB limit).
# Skip this cell if BIN_PATH already exists (shown below).

import multiprocessing as _mp, struct as _struct2, gzip as _gz2, time as _time2
import numpy as _np2

BIN_PATH    = Path('/kaggle/working/training.bin')
MAX_RECORDS = 140_000_000    # ~18 GB
_MAGIC2     = b'HKAV2BIN'
_RECSIZE2   = 32*2 + 32*2 + 2   # 130 bytes
_BATCH2     = 50_000             # lines per worker task

def _prep_batch(args):
    """Self-contained worker: parses raw lines → packed binary. No imports from other cells."""
    lines, cp_scale = args
    import math, numpy as np
    FT=45056; MA=32; NS=11
    _P={'p':1,'n':2,'b':3,'r':4,'q':5,'k':6}
    def _bd(b):
        p,sq=[],56
        for ch in b:
            if ch=='/': sq-=16; continue
            if ch.isdigit(): sq+=int(ch); continue
            p.append((sq,0 if ch.isupper() else 1,_P[ch.lower()])); sq+=1
        return p
    def _fi(ks,ps,pt,o):
        if o and pt==6: raise ValueError
        return ks*(64*NS)+ps*NS+((pt-1) if o else (5+pt-1))
    def _ff(fen):
        b,s=fen.split()[0],fen.split()[1]; pcs=_bd(b); sc=0 if s=='w' else 1
        ks=[None,None]
        for sq,c,pt in pcs:
            if pt==6: ks[c]=sq
        if None in ks: raise ValueError
        ix=[[],[]]
        for pp in(0,1):
            kq=ks[pp] if pp==0 else ks[pp]^56
            for sq,c,pt in pcs:
                if c==pp and pt==6: continue
                ix[pp].append(_fi(kq,sq if pp==0 else sq^56,pt,c==pp))
        return (ix[0],ix[1]) if sc==0 else (ix[1],ix[0])
    def _pl(line):
        line=line.strip()
        if not line or line.startswith('#'): return None
        p=line.split(';')
        if len(p)==4:
            try: fen=p[0].strip(); wp=float(p[1])+0.5*float(p[2])
            except: return None
        elif len(p)==2:
            try: fen=p[0].strip(); wp=1.0/(1.0+math.exp(-float(p[1])/cp_scale))
            except: return None
        else: return None
        try: us,them=_ff(fen)
        except: return None
        ua=np.full(MA,FT,dtype=np.uint16); ta=np.full(MA,FT,dtype=np.uint16)
        ua[:min(len(us),MA)]=us[:MA]; ta[:min(len(them),MA)]=them[:MA]
        return ua,ta,np.float16(wp)
    dt=np.dtype([('us',np.uint16,(MA,)),('them',np.uint16,(MA,)),('tgt',np.float16)])
    recs=[r for line in lines for r in [_pl(line)] if r]
    if not recs: return b''
    buf=np.empty(len(recs),dtype=dt)
    for i,(u,t,w) in enumerate(recs): buf[i]['us']=u; buf[i]['them']=t; buf[i]['tgt']=w
    return buf.tobytes()

if BIN_PATH.exists():
    with open(BIN_PATH,'rb') as _f: _f.read(8); _n=_struct2.unpack('<q',_f.read(8))[0]
    print(f'binary exists: {BIN_PATH}')
    print(f'  {_n:,} positions  {BIN_PATH.stat().st_size/1e9:.1f} GB — skipping preprocessing')
else:
    def _batches2(path):
        op=_gz2.open if str(path).endswith('.gz') else open; batch=[]
        with op(path,'rt',encoding='utf-8') as f:
            for line in f:
                s=line.strip()
                if s and not s.startswith('#'):
                    batch.append(line)
                    if len(batch)==_BATCH2: yield (batch,410.0); batch=[]
        if batch: yield (batch,410.0)

    print(f'preprocessing {DATA_PATH} → {BIN_PATH}')
    print(f'cap: {MAX_RECORDS:,} records (~{MAX_RECORDS*_RECSIZE2/1e9:.1f} GB)')
    total=0; t0=_time2.time()
    with open(BIN_PATH,'wb') as fout:
        fout.write(_MAGIC2); fout.write(_struct2.pack('<q',0))
        with _mp.Pool(4) as pool:
            for data in pool.imap(_prep_batch, _batches2(DATA_PATH), chunksize=4):
                if not data: continue
                nr=len(data)//_RECSIZE2
                if total+nr>MAX_RECORDS: keep=MAX_RECORDS-total; data=data[:keep*_RECSIZE2]; nr=keep
                fout.write(data); total+=nr
                el=_time2.time()-t0
                print(f'\r  {total:>12,}/{MAX_RECORDS:,}  {total*_RECSIZE2/1e9:.1f} GB  '
                      f'{total/el/1e3 if el else 0:.0f}k pos/s', end='')
                if total>=MAX_RECORDS: break
    with open(BIN_PATH,'r+b') as f: f.seek(8); f.write(_struct2.pack('<q',total))
    print(f'\ndone: {total:,} positions in {_time2.time()-t0:.0f}s — '
          f'{BIN_PATH.stat().st_size/1e9:.1f} GB')

In [ ]:
# Model -- float mirror of the C++ inference path in src/nnue.cpp.
class ClippedReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, 0.0, 1.0)

class HalfKAv2Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Last embedding row is the padding sentinel: zeroed at init,
        # pinned to zero after every step so padded slots contribute 0.
        self.ft = nn.Embedding(FT_IN_FEATURES + 1, FT_OUT,
                               padding_idx=FT_IN_FEATURES)
        nn.init.uniform_(self.ft.weight, -0.01, 0.01)
        with torch.no_grad():
            self.ft.weight[FT_IN_FEATURES].zero_()
        self.ft_bias = nn.Parameter(torch.zeros(FT_OUT))
        self.l1  = nn.Linear(L1_IN, L1_OUT)
        self.l2  = nn.Linear(L1_OUT, L2_OUT)
        self.l3  = nn.Linear(L2_OUT, L3_OUT)
        self.act = ClippedReLU()

    def forward(self, us_idx, them_idx):
        us   = self.ft(us_idx).sum(dim=1)   + self.ft_bias
        them = self.ft(them_idx).sum(dim=1) + self.ft_bias
        x = torch.cat([self.act(us), self.act(them)], dim=1)
        x = self.act(self.l1(x))
        x = self.act(self.l2(x))
        return self.l3(x)

    def export_state_dict_for_quantization(self):
        # Repack the Embedding into a Linear-style weight matrix
        # [FT_OUT, FT_IN_FEATURES] so convert_halfkav2_nnue.py reads it cleanly.
        sd = {}
        ft_w = self.ft.weight.detach()
        sd['ft.weight'] = ft_w[:FT_IN_FEATURES].T.contiguous()
        sd['ft.bias']   = self.ft_bias.detach()
        sd['l1.weight'] = self.l1.weight.detach()
        sd['l1.bias']   = self.l1.bias.detach()
        sd['l2.weight'] = self.l2.weight.detach()
        sd['l2.bias']   = self.l2.bias.detach()
        sd['l3.weight'] = self.l3.weight.detach()
        sd['l3.bias']   = self.l3.bias.detach()
        return sd

In [ ]:
# Training loop. BCE on sigmoid win-prob. Adam + linear warmup + cosine schedule.
import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count() if device.type == 'cuda' else 0
print(f'training on {device}  ({n_gpu} GPU{"s" if n_gpu != 1 else ""})')
if n_gpu > 1:
    for i in range(n_gpu):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

BIN_PATH = Path('/kaggle/working/training.bin')   # also set by preprocessing cell
_use_bin = (BIN_PATH.exists() and
            open(BIN_PATH, 'rb').read(8) == b'HKAV2BIN')

t0 = time.time()
if _use_bin:
    print(f'binary dataset: {BIN_PATH}')
    val_ds   = HalfKAv2BinaryDataset(BIN_PATH, end=CONFIG['val_size'])
    train_ds = HalfKAv2BinaryDataset(BIN_PATH, start=len(val_ds))
    n_val    = len(val_ds)
    print(f'val: {n_val:,}  train: {len(train_ds):,} positions (random-access, no FEN parsing)')
    _shuffle_train = True
else:
    print('binary not found — using streaming text (run preprocessing cell first for full speed)')
    print(f'loading {CONFIG["val_size"]:,} val positions into memory ...')
    val_ds = HalfKAv2Dataset(DATA_PATH, max_lines=CONFIG['val_size'],
                             cp_scale=CONFIG['cp_scale'])
    n_val  = len(val_ds)
    train_ds = HalfKAv2StreamDataset(
        DATA_PATH, skip_lines=n_val, max_lines=CONFIG['max_lines'],
        cp_scale=CONFIG['cp_scale'], shuffle_buffer=CONFIG['shuffle_buffer'])
    suffix = f" (max {CONFIG['max_lines']:,}/epoch)" if CONFIG['max_lines'] else ''
    print(f'val: {n_val:,} positions (in memory)  train: streaming{suffix}')
    _shuffle_train = False

print(f'setup in {time.time()-t0:.1f}s')

eff_batch   = CONFIG['batch_size'] * max(n_gpu, 1)
eff_workers = CONFIG['num_workers']   # total workers; DataParallel uses one DataLoader
loader = DataLoader(train_ds, batch_size=eff_batch, shuffle=_shuffle_train,
                    num_workers=eff_workers, collate_fn=collate,
                    drop_last=True, pin_memory=(device.type == 'cuda'),
                    persistent_workers=(eff_workers > 0),
                    prefetch_factor=(4 if eff_workers > 0 else None))
val_loader = DataLoader(val_ds, batch_size=eff_batch, shuffle=False,
                        num_workers=max(1, eff_workers // 2), collate_fn=collate,
                        drop_last=False, pin_memory=(device.type == 'cuda'),
                        persistent_workers=True, prefetch_factor=4)

net = HalfKAv2Net().to(device)
if n_gpu > 1:
    net = nn.DataParallel(net)
    print(f'DataParallel across {n_gpu} GPUs, effective batch {eff_batch}')
base_net = net.module if isinstance(net, nn.DataParallel) else net
opt    = torch.optim.Adam(net.parameters(), lr=CONFIG['lr'],
                          weight_decay=CONFIG['weight_decay'])
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])

out_path = Path(CONFIG['out_path'])
out_path.parent.mkdir(parents=True, exist_ok=True)

step = 0
for epoch in range(CONFIG['epochs']):
    net.train()
    ep_loss = 0.0; nbatch = 0
    for us, them, target in loader:
        us, them, target = us.to(device), them.to(device), target.to(device)
        if step < CONFIG['warmup_steps']:
            scale = (step + 1) / max(1, CONFIG['warmup_steps'])
            for g in opt.param_groups:
                g['lr'] = CONFIG['lr'] * scale
        logits = net(us, them)
        loss = F.binary_cross_entropy_with_logits(logits.squeeze(1), target.squeeze(1))
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            base_net.ft.weight[FT_IN_FEATURES].zero_()
        ep_loss += loss.item(); nbatch += 1; step += 1
        if step % 200 == 0:
            print(f'  epoch {epoch} step {step} loss {loss.item():.5f}')

    net.eval()
    vl_sum = 0.0; vmae_sum = 0.0; vcount = 0
    with torch.no_grad():
        for us, them, target in val_loader:
            us, them, target = us.to(device), them.to(device), target.to(device)
            logits = net(us, them)
            vl_sum  += F.binary_cross_entropy_with_logits(
                logits.squeeze(1), target.squeeze(1)).item() * target.shape[0]
            pred_cp  = logits.squeeze(1) * CONFIG['cp_scale']
            tgt_cp   = torch.special.logit(target.squeeze(1).clamp(1e-6, 1-1e-6)) * CONFIG['cp_scale']
            vmae_sum += (pred_cp - tgt_cp).abs().sum().item()
            vcount   += target.shape[0]
    cosine.step()
    print(f'epoch {epoch} train_loss {ep_loss/max(nbatch,1):.5f} '
          f'val_loss {vl_sum/max(1,vcount):.5f} '
          f'val_mae_cp {vmae_sum/max(1,vcount):.1f} '
          f'(lr next: {cosine.get_last_lr()[0]:.6f})')

    ckpt = out_path.with_name(out_path.stem + f'_epoch{epoch}' + out_path.suffix)
    torch.save(base_net.export_state_dict_for_quantization(), ckpt)
    print(f'saved {ckpt} ({ckpt.stat().st_size/1e6:.1f} MB)')

In [ ]:
# Training loop. BCE on sigmoid win-prob. Adam + linear warmup + cosine schedule.
# Val set is the first val_size lines (in memory). Train streams the rest.
import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count() if device.type == 'cuda' else 0
print(f'training on {device}  ({n_gpu} GPU{"s" if n_gpu != 1 else ""})')
if n_gpu > 1:
    for i in range(n_gpu):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

t0 = time.time()
print(f'loading {CONFIG["val_size"]:,} val positions into memory ...')
val_ds = HalfKAv2Dataset(DATA_PATH, max_lines=CONFIG['val_size'],
                         cp_scale=CONFIG['cp_scale'])
n_val  = len(val_ds)
train_ds = HalfKAv2StreamDataset(
    DATA_PATH, skip_lines=n_val, max_lines=CONFIG['max_lines'],
    cp_scale=CONFIG['cp_scale'], shuffle_buffer=CONFIG['shuffle_buffer'])
suffix = f" (max {CONFIG['max_lines']:,}/epoch)" if CONFIG['max_lines'] else ''
print(f'val: {n_val:,} positions (in memory)  train: streaming{suffix}')
print(f'setup in {time.time()-t0:.1f}s')

eff_batch   = CONFIG['batch_size'] * max(n_gpu, 1)
eff_workers = CONFIG['num_workers'] * max(n_gpu, 1)
loader = DataLoader(train_ds, batch_size=eff_batch, shuffle=False,
                    num_workers=eff_workers, collate_fn=collate,
                    drop_last=True, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_ds, batch_size=eff_batch, shuffle=False,
                        num_workers=max(1, eff_workers//2), collate_fn=collate,
                        drop_last=False, pin_memory=(device.type == 'cuda'))

net = HalfKAv2Net().to(device)
if n_gpu > 1:
    net = nn.DataParallel(net)
    print(f'DataParallel across {n_gpu} GPUs, effective batch {eff_batch}')
base_net = net.module if isinstance(net, nn.DataParallel) else net
opt    = torch.optim.Adam(net.parameters(), lr=CONFIG['lr'],
                          weight_decay=CONFIG['weight_decay'])
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])

step = 0
for epoch in range(CONFIG['epochs']):
    net.train()
    ep_loss = 0.0; nbatch = 0
    for us, them, target in loader:
        us, them, target = us.to(device), them.to(device), target.to(device)
        if step < CONFIG['warmup_steps']:
            scale = (step + 1) / max(1, CONFIG['warmup_steps'])
            for g in opt.param_groups:
                g['lr'] = CONFIG['lr'] * scale
        logits = net(us, them)
        loss = F.binary_cross_entropy_with_logits(logits.squeeze(1), target.squeeze(1))
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            base_net.ft.weight[FT_IN_FEATURES].zero_()
        ep_loss += loss.item(); nbatch += 1; step += 1
        if step % 200 == 0:
            print(f'  epoch {epoch} step {step} loss {loss.item():.5f}')

    net.eval()
    vl_sum = 0.0; vmae_sum = 0.0; vcount = 0
    with torch.no_grad():
        for us, them, target in val_loader:
            us, them, target = us.to(device), them.to(device), target.to(device)
            logits = net(us, them)
            vl_sum  += F.binary_cross_entropy_with_logits(
                logits.squeeze(1), target.squeeze(1)).item() * target.shape[0]
            pred_cp  = logits.squeeze(1) * CONFIG['cp_scale']
            tgt_cp   = torch.special.logit(target.squeeze(1).clamp(1e-6, 1-1e-6)) * CONFIG['cp_scale']
            vmae_sum += (pred_cp - tgt_cp).abs().sum().item()
            vcount   += target.shape[0]
    cosine.step()
    print(f'epoch {epoch} train_loss {ep_loss/max(nbatch,1):.5f} '
          f'val_loss {vl_sum/max(1,vcount):.5f} '
          f'val_mae_cp {vmae_sum/max(1,vcount):.1f} '
          f'(lr next: {cosine.get_last_lr()[0]:.6f})')

In [ ]:
# Save state_dict in the layout convert_halfkav2_nnue.py expects.
out_path = Path(CONFIG['out_path'])
out_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(base_net.export_state_dict_for_quantization(), out_path)
size_mb = out_path.stat().st_size / 1e6
print(f'saved {out_path} ({size_mb:.1f} MB)')
print()
print('Next: download halfkav2.pt and locally run')
print('  python scripts/convert_halfkav2_nnue.py from-torch \\')
print(f'      --state-dict data/halfkav2.pt \\')
print(f'      --out data/eclipse.nnue \\')
print(f'      --output-cp-per-unit {CONFIG["cp_scale"]}')

## Saving the artifact off Kaggle

1. Click **Save Version → Save & Run All**. The notebook output (including `/kaggle/working/halfkav2.pt`) is captured on the version that finishes successfully.
2. Open the saved version → **Output** tab → click `halfkav2.pt` → Download.
3. Locally, drop it into `data/` and run the convert command above. See `KAGGLE.md` for verification steps and saturation-warning interpretation.